# Task 1: Web Scraping – TechReads Retail Analytics
**Module:** CMP-X304-0 Data Engineering  
**Target Website:** books.toscrape.com  
**Fields Extracted:** Title, Price, Star Rating, Availability, Category  
**Output:** techreads_books.csv

## Step 1: Import Required Libraries

In [1]:
# requests: used to download the webpage HTML content
import requests

# BeautifulSoup: used to parse and navigate the HTML structure
from bs4 import BeautifulSoup

# pandas: used to organise extracted data into a structured DataFrame and export to CSV
import pandas as pd

## Step 2: Define Target URL and Rating Conversion

In [2]:
# The target URL – page 1 of books.toscrape.com
# This site is designed for web scraping practice and is safe to use
BASE_URL = 'https://books.toscrape.com/'

# The site stores star ratings as words (e.g. 'Three'), not numbers
# This dictionary maps word ratings to numeric values for easier analysis
RATING_MAP = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}

## Step 3: Download and Parse the Webpage

In [3]:
# Send an HTTP GET request to the target URL
response = requests.get(BASE_URL)

# Check that the request was successful (status code 200 = OK)
print(f'Status Code: {response.status_code}')

# Parse the downloaded HTML using BeautifulSoup
# 'html.parser' is Python's built-in HTML parser – no extra install needed
soup = BeautifulSoup(response.text, 'html.parser')

print('Page downloaded and parsed successfully.')

Status Code: 200
Page downloaded and parsed successfully.


## Step 4: Extract Category from Sidebar

In [4]:
# The homepage shows books from all categories mixed together
# We extract the category list from the sidebar navigation for reference
# Note: books.toscrape.com does not provide Author or Year fields
# Category and Availability are used as substitute descriptive fields

# Each book article element contains a 'data-track' class or is under <ol class='row'>
# We will assign category as 'General' since homepage mixes all categories
# If you scrape a specific category page, you can extract the exact category name

# Get the active category name from the sidebar (if on a category page)
# On the homepage, this will return None
active_category = soup.find('li', class_='active')
if active_category:
    category_name = active_category.text.strip()
else:
    category_name = 'Mixed'

print(f'Current page category: {category_name}')

Current page category: All products


## Step 5: Scrape All Books on Page 1

In [5]:
# Find all book entries on the page
# Each book is contained within an <article class='product_pod'> element
books_html = soup.find_all('article', class_='product_pod')

print(f'Number of books found on page: {len(books_html)}')

# Initialise an empty list to store each book's data as a dictionary
books_data = []

for book in books_html:
    
    # --- Extract Title ---
    # The title is stored in the 'title' attribute of the <a> tag inside <h3>
    # Using 'title' attribute gives the full title (the text is sometimes truncated)
    title = book.find('h3').find('a')['title']
    
    # --- Extract Price ---
    # Price is inside <p class='price_color'>
    # We remove the '£' symbol and convert to a float for numeric storage
    price_text = book.find('p', class_='price_color').text.strip()
    price = float(price_text.replace('£', '').replace('Â', '').strip())
    
    # --- Extract Star Rating ---
    # Rating is stored as a CSS class on <p class='star-rating Three'>
    # We extract the second class name (e.g. 'Three') and convert using RATING_MAP
    rating_word = book.find('p', class_='star-rating')['class'][1]
    star_rating = RATING_MAP.get(rating_word, 0)
    
    # --- Extract Availability ---
    # Availability is inside <p class='instock availability'>
    availability = book.find('p', class_='availability').text.strip()
    
    # --- Category ---
    # Homepage mixes all categories, so we label as 'Mixed'
    # This field replaces 'Author' which is not available on this site
    category = category_name
    
    # Append this book's data as a dictionary to our list
    books_data.append({
        'title': title,
        'price': price,
        'star_rating': star_rating,
        'availability': availability,
        'category': category
    })

print(f'Successfully extracted data for {len(books_data)} books.')

Number of books found on page: 20
Successfully extracted data for 20 books.


## Step 6: Structure Data Using Pandas

In [6]:
# Convert the list of dictionaries into a pandas DataFrame
# DataFrame provides a tabular structure ideal for data inspection and CSV export
df = pd.DataFrame(books_data)

# Display the first few rows to verify extraction was successful
print(f'DataFrame shape: {df.shape}')  # (rows, columns)
df.head(10)

DataFrame shape: (20, 5)


,title,price,star_rating,availability,category
0,A Light in the Attic,51.77,3,In stock,All products
1,Tipping the Velvet,53.74,1,In stock,All products
2,Soumission,50.10,1,In stock,All products
3,Sharp Objects,47.82,4,In stock,All products
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,All products
5,The Requiem Red,22.65,1,In stock,All products
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,All products
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,All products
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,In stock,All products
9,The Black Maria,52.15,1,In stock,All products


In [7]:
# Check data types to ensure numeric fields are stored correctly
print(df.dtypes)

# Check for any missing values
print('\nMissing values per column:')
print(df.isnull().sum())

title               str
price           float64
star_rating       int64
availability        str
category            str
dtype: object

Missing values per column:
title           0
price           0
star_rating     0
availability    0
category        0
dtype: int64


## Step 7: Save to CSV

In [8]:
# Save the DataFrame to a CSV file named as required by the assessment brief
# index=False prevents pandas from writing the row index as an extra column
# encoding='utf-8-sig' ensures special characters display correctly in Excel
output_filename = 'techreads_books.csv'
df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f'Data successfully saved to {output_filename}')
print(f'Total books saved: {len(df)}')

Data successfully saved to techreads_books.csv
Total books saved: 20


## Step 8: Verify CSV Output

In [9]:
# Read the saved CSV back to confirm it was written correctly
df_verify = pd.read_csv(output_filename)

print('CSV file verified. Contents:')
print(df_verify.to_string())

CSV file verified. Contents:
                                                                                             title  price  star_rating availability      category
0                                                                             A Light in the Attic  51.77            3     In stock  All products
1                                                                               Tipping the Velvet  53.74            1     In stock  All products
2                                                                                       Soumission  50.10            1     In stock  All products
3                                                                                    Sharp Objects  47.82            4     In stock  All products
4                                                            Sapiens: A Brief History of Humankind  54.23            5     In stock  All products
5                                                                                  The Requiem 